In [36]:
%pip install scikit-learn
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#### 2. Preparação dos Dados (Split Treino/Teste e Estandardização)
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Carregar o dataset tratado gerado pelo primeiro notebook
df_clean = pd.read_csv('../data/dataset_trabalhado.csv')



LIMITE_PRECO = 10_000_000  # 1 milhão de euros para limitar, 10 milhões para nao limitar

# --- Opção A: tens um único DataFrame (ex.: df) com a coluna do preço ---
col_preco = "price_eur"  # substitui pelo nome real da coluna de preço

linhas_antes = len(df_clean)
df_filtrado = df_clean[df_clean[col_preco] <= LIMITE_PRECO].copy()
linhas_removidas = linhas_antes - len(df_filtrado)

print(f"Linhas antes: {linhas_antes} | Linhas removidas (> 1M €): {linhas_removidas} | Restantes: {len(df_filtrado)}")

# 1. Separar as características (X) do preço (y)
X = df_filtrado.drop(columns=["price_eur", "id"], errors="ignore")
y = df_filtrado["price_eur"]

# 2. Definir grupos de colunas (numéricas, binárias e categóricas dummy)
num_cols = [
    "area_m2", "latitude", "longitude",
    "bedrooms", "bathrooms", "floor", "num_floors"
]
bin_cols = ["has_garage", "has_garden"]
cat_cols = [c for c in X.columns if c.startswith("property_type_")]

# 3. Dividir em 80% treino e 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Criar o ColumnTransformer (estandardizar numéricas e manter binárias/dummies)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols + cat_cols),
    ]
)

# 5. Ajustar apenas no treino e transformar treino e teste
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# 6. Guardar o preprocessor para transformar imóveis novos no futuro
joblib.dump(preprocessor, "preprocessor.joblib")

print("Dimensão de X_train pronto:", X_train_scaled.shape)
print("Exemplo de preços intactos (y_train):", y_train.head(3).values)
print(f"Treino: {len(X_train)} imóveis | Teste: {len(X_test)} imóveis")

Linhas antes: 6749 | Linhas removidas (> 1M €): 790 | Restantes: 5959
Dimensão de X_train pronto: (4767, 11)
Exemplo de preços intactos (y_train): [320000 425000 290000]
Treino: 4767 imóveis | Teste: 1192 imóveis


In [ ]:
#### 3. Treino e Avaliação do Modelo HistGradientBoostingRegressor
import joblib
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Instanciar e treinar (fit) o modelo com os dados de treino estandardizados
modelo = HistGradientBoostingRegressor(random_state=42)
modelo.fit(X_train_scaled, y_train)

# 2. Fazer previsões e avaliar no conjunto de teste
y_pred = modelo.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Erro percentual absoluto por imóvel (protegido contra y_test == 0)
y_true_arr = np.asarray(y_test, dtype=float)
y_pred_arr = np.asarray(y_pred, dtype=float)
mask_validos = y_true_arr != 0

erros_pct = np.abs(y_true_arr[mask_validos] - y_pred_arr[mask_validos]) / np.abs(y_true_arr[mask_validos]) * 100
mape = np.mean(erros_pct)     # Média do erro percentual
mdape = np.median(erros_pct)  # Mediana do erro percentual (menos sensível a casos extremos)

# 3. Guardar o modelo treinado
joblib.dump(modelo, "modelo_imoveis.joblib")

print(f"MAE (Erro Médio Absoluto):          {mae:,.2f} €")
print(f"MAPE (Erro Percentual Médio):       {mape:.2f}%")
print(f"MdAPE (Mediana do Erro Percentual): {mdape:.2f}%")
print(f"R² Score:                           {r2:.4f}")

MAE (Erro Médio Absoluto):          66,270.01 €
MAPE (Erro Percentual Médio):       15.20%
MdAPE (Mediana do Erro Percentual): 10.78%
R² Score:                           0.7721


In [ ]:
#### Comparação dos modelos com um que é sempre a mediana ####

import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# Substitui pelos nomes reais das tuas variáveis:
# y_train -> preços reais do conjunto de treino
# y_test   -> preços reais do conjunto de validação/teste
# y_pred  -> previsões do teu modelo em y_test

# 1. Baseline: prever sempre a mediana do treino
mediana_treino = np.median(y_train)
y_pred_baseline = np.full(shape=len(y_test), fill_value=mediana_treino)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

# 2. Métricas do teu modelo
mae_modelo = mean_absolute_error(y_test, y_pred)
r2_modelo = r2_score(y_test, y_pred)

# 3. Comparação
reducao_mae = mae_baseline - mae_modelo
reducao_pct = (reducao_mae / mae_baseline) * 100
mediana_val = np.median(y_test)
media_val = np.mean(y_test)

print(f"Mediana do preço (validação): {mediana_val:,.2f} €")
print(f"Média do preço (validação):   {media_val:,.2f} €")
print("-" * 45)
print(f"Baseline (mediana treino = {mediana_treino:,.2f} €):")
print(f"  MAE: {mae_baseline:,.2f} € | R²: {r2_baseline:.4f}")
print(f"Teu modelo:")
print(f"  MAE: {mae_modelo:,.2f} € | R²: {r2_modelo:.4f}")
print("-" * 45)
print(f"Redução de MAE vs. baseline: {reducao_mae:,.2f} € ({reducao_pct:.1f}%)")

Mediana do preço (validação): 385,000.00 €
Média do preço (validação):   448,169.23 €
---------------------------------------------
Baseline (mediana treino = 385,000.00 €):
  MAE: 155,724.74 € | R²: -0.0995
Teu modelo:
  MAE: 65,199.11 € | R²: 0.7804
---------------------------------------------
Redução de MAE vs. baseline: 90,525.62 € (58.1%)


In [ ]:
#### Printer dos maiores erros absolutos ####


y_real = np.asarray(y_test)
y_prev = np.asarray(y_pred)

erro_abs = np.abs(y_real - y_prev)
erro_rel_pct = np.where(y_real != 0, (erro_abs / np.abs(y_real)) * 100, np.nan)

df_erros = pd.DataFrame({
    "preco_real": y_real,
    "preco_previsto": y_prev,
    "erro_abs_eur": erro_abs,
    "erro_rel_pct": erro_rel_pct,
}, index=getattr(X_test, "index", range(len(y_real))))

# Se X_test for um DataFrame, junta as colunas de características para inspeção
if isinstance(X_test, pd.DataFrame):
    df_erros = pd.concat([df_erros, X_test.loc[df_erros.index]], axis=1)

# Top 15 maiores erros absolutos
top_15_erros = df_erros.sort_values("erro_abs_eur", ascending=False).head(15)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print(top_15_erros.round(2))

      preco_real  preco_previsto  erro_abs_eur  erro_rel_pct  area_m2  latitude  longitude  bedrooms  bathrooms  floor  num_floors  has_garage  has_garden  property_type_apartment  property_type_house
5235      350000       752773.44     402773.44        115.08      273     41.20      -8.69         4        4.0    0.0         1.0           0           0                        0                    1
2756      990000       588711.80     401288.20         40.53      175     41.12      -8.65         3        3.0    2.0         2.0           1           1                        0                    1
1721     1000000       626157.03     373842.97         37.38      185     41.13      -8.66         3        3.0    1.0         1.0           1           1                        1                    0
2654      185000       553445.34     368445.34        199.16      148     41.14      -8.62         2        3.0    0.0         1.0           0           0                        0                 

In [ ]:
#### 3. Treino e Avaliação do Modelo XGBRegressor

import numpy as np
from sklearn.metrics import mean_absolute_error, make_scorer, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor


def calcular_mape(y_true, y_pred):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    mask = y_true_arr != 0
    return np.mean(np.abs(y_true_arr[mask] - y_pred_arr[mask]) / np.abs(y_true_arr[mask])) * 100


def criar_scorer_mae(usar_log):
    def mae_em_euros(y_true, y_pred):
        if usar_log:
            y_true = np.expm1(y_true)
            y_pred = np.expm1(y_pred)
        return mean_absolute_error(y_true, y_pred)

    return make_scorer(mae_em_euros, greater_is_better=False)


param_dist = {
    "n_estimators": [300, 500, 800, 1200],
    "learning_rate": [0.01, 0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.1, 1.0, 5.0],
    "reg_lambda": [1.0, 5.0, 10.0, 20.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
resultados = {}

for usar_log in [False, True]:
    nome_modo = "COM ln (log1p)" if usar_log else "SEM ln (euros)"
    y_alvo = np.log1p(y_train) if usar_log else y_train

    modelo_base = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    )

    busca = RandomizedSearchCV(
        estimator=modelo_base,
        param_distributions=param_dist,
        n_iter=25,  # aumenta para 40-60 se correr rápido no teu computador
        scoring=criar_scorer_mae(usar_log),
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )

    busca.fit(X_train, y_alvo)
    melhor_modelo = busca.best_estimator_

    pred_teste = melhor_modelo.predict(X_test)
    if usar_log:
        pred_teste = np.expm1(pred_teste)

    mae_cv = -busca.best_score_
    mae_teste = mean_absolute_error(y_test, pred_teste)
    mape_teste = calcular_mape(y_test, pred_teste)
    r2_teste = r2_score(y_test, pred_teste)

    resultados[nome_modo] = {
        "modelo": melhor_modelo,
        "params": busca.best_params_,
        "pred": pred_teste,
        "mae_cv": mae_cv,
        "mae_teste": mae_teste,
        "mape_teste": mape_teste,
        "r2_teste": r2_teste,
    }

    print(f"=== {nome_modo} ===")
    print("Melhores hiperparâmetros:", busca.best_params_)
    print(f"MAE médio em CV: {mae_cv:,.2f} €")
    print(f"MAE no teste:    {mae_teste:,.2f} €")
    print(f"MAPE no teste:   {mape_teste:.2f}%")
    print(f"R² no teste:     {r2_teste:.4f}\n")

print("-" * 55)
print("RESUMO COMPARATIVO NO TESTE:")
for modo, res in resultados.items():
    print(f"{modo:18s} | MAE: {res['mae_teste']:10,.2f} € | MAPE: {res['mape_teste']:6.2f}% | R²: {res['r2_teste']:.4f}")

=== SEM ln (euros) ===
Melhores hiperparâmetros: {'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.9}
MAE médio em CV: 61,786.50 €
MAE no teste:    63,084.91 €
MAPE no teste:   14.38%
R² no teste:     0.7857

=== COM ln (log1p) ===
Melhores hiperparâmetros: {'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.9}
MAE médio em CV: 61,559.54 €
MAE no teste:    61,923.24 €
MAPE no teste:   13.69%
R² no teste:     0.7817

-------------------------------------------------------
RESUMO COMPARATIVO NO TESTE:
SEM ln (euros)     | MAE:  63,084.91 € | MAPE:  14.38% | R²: 0.7857
COM ln (log1p)     | MAE:  61,923.24 € | MAPE:  13.69% | R²: 0.7817
